# 05 - Vista previa de resultados

Este notebook no sustituye a Power BI. Sirve para ver como va quedando la historia visual del TFM antes de montar el dashboard final.

La logica de negocio que queremos mostrar es:

- volumen de ventas, costes y margen;
- rentabilidad por canal;
- categorias/productos que explican el margen;
- prediccion de unidades vendidas;
- riesgo de stock.

## 0. Preparacion

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

from conexion_sql import read_sql
from eda_utils import asegurar_directorio
from visualizacion_utils import guardar_figura

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

OUT_DATOS = asegurar_directorio(PROJECT_ROOT / "outputs" / "datos")
OUT_GRAFICOS = asegurar_directorio(PROJECT_ROOT / "outputs" / "graficos")

## 1. KPIs ejecutivos

In [ ]:
ventas = read_sql("SELECT * FROM dbo.vw_ventas_enriquecidas")
ventas["fecha"] = pd.to_datetime(ventas["fecha"])

kpis = pd.DataFrame({
    "KPI": ["Ventas netas", "Costes totales", "Margen", "Margen %", "Pedidos", "Unidades"],
    "Valor": [
        ventas["ventas_netas"].sum(),
        ventas["costes_totales"].sum(),
        ventas["margen"].sum(),
        ventas["margen"].sum() / ventas["ventas_netas"].sum(),
        ventas["id_pedido"].nunique(),
        ventas["unidades_vendidas"].sum(),
    ]
})

display(kpis)
kpis.to_csv(OUT_DATOS / "preview_kpis.csv", index=False)

## 2. Rentabilidad por canal

In [ ]:
canal = (
    ventas.groupby("canal", as_index=False)
    .agg(ventas_netas=("ventas_netas", "sum"), margen=("margen", "sum"), comisiones=("comision", "sum"))
)
canal["margen_pct"] = canal["margen"] / canal["ventas_netas"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=canal.sort_values("ventas_netas", ascending=False), x="ventas_netas", y="canal", ax=axes[0], color="#4C78A8")
axes[0].set_title("Ventas netas por canal")
axes[0].set_xlabel("Ventas netas")
axes[0].set_ylabel("")

sns.barplot(data=canal.sort_values("margen_pct", ascending=False), x="margen_pct", y="canal", ax=axes[1], color="#F58518")
axes[1].set_title("Margen % por canal")
axes[1].set_xlabel("Margen %")
axes[1].set_ylabel("")

guardar_figura("preview_rentabilidad_canal.png", OUT_GRAFICOS)
plt.show()

## 3. Top categorias por margen

In [ ]:
categoria = (
    ventas.groupby("categoria", as_index=False)
    .agg(ventas_netas=("ventas_netas", "sum"), margen=("margen", "sum"), unidades=("unidades_vendidas", "sum"))
)
categoria["margen_pct"] = categoria["margen"] / categoria["ventas_netas"]
categoria_top = categoria.sort_values("margen", ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(data=categoria_top, x="margen", y="categoria", color="#54A24B")
plt.title("Top categorias por margen total")
plt.xlabel("Margen")
plt.ylabel("")
guardar_figura("preview_top_categorias_margen.png", OUT_GRAFICOS)
plt.show()

## 4. Prediccion de ventas si ya existe

In [ ]:
ruta_pred = OUT_DATOS / "predicciones_ml_ventas.csv"
if ruta_pred.exists():
    pred = pd.read_csv(ruta_pred, parse_dates=["fecha"])
    metricas = pd.read_csv(OUT_DATOS / "metricas_ml_ventas.csv")
    mejor_modelo = metricas.sort_values("RMSE").iloc[0]["modelo"]
    pred_best = pred[pred["modelo"] == mejor_modelo]
    pred_fecha = pred_best.groupby("fecha", as_index=False).agg(real=("real", "sum"), prediccion=("prediccion", "sum"))

    display(metricas.sort_values("RMSE"))

    plt.figure(figsize=(14, 5))
    sns.lineplot(data=pred_fecha, x="fecha", y="real", label="Real")
    sns.lineplot(data=pred_fecha, x="fecha", y="prediccion", label="Prediccion")
    plt.title(f"Preview prediccion ventas - {mejor_modelo}")
    plt.xticks(rotation=45)
    guardar_figura("preview_prediccion_ventas.png", OUT_GRAFICOS)
    plt.show()
else:
    print("Todavia no existen predicciones. Ejecuta primero el notebook 03.")

## 5. Riesgo de stock

In [ ]:
stock = read_sql("SELECT * FROM dbo.vw_riesgo_stock")

if "nivel_riesgo_stock" in stock.columns:
    stock_resumen = stock["nivel_riesgo_stock"].value_counts().reset_index()
    stock_resumen.columns = ["nivel_riesgo_stock", "productos_almacen"]
    display(stock_resumen)

    plt.figure(figsize=(8, 4))
    sns.barplot(data=stock_resumen, x="productos_almacen", y="nivel_riesgo_stock", color="#E45756")
    plt.title("Productos/almacen por nivel de riesgo de stock")
    plt.xlabel("Registros")
    plt.ylabel("")
    guardar_figura("preview_riesgo_stock.png", OUT_GRAFICOS)
    plt.show()
else:
    display(stock.head())

## 6. Lectura de negocio

Esta vista previa deberia contar una historia sencilla:

1. Cuanto vendemos y cuanto margen real queda.
2. Que canales son mas rentables y cuales consumen margen.
3. Que categorias aportan mas beneficio.
4. Si el modelo predictivo anticipa razonablemente la demanda.
5. Donde hay riesgo de stock para actuar antes de perder ventas.